# 04. Embedding: превращаем ID токенов в векторы

В прошлом блокноте мы превратили слова в целые числа — token ID. Но ID сам по себе не несёт информации о значении слова. Например, ID 7 не означает, что слово «лучше» или «хуже», чем слово с ID 3.

Transformer работает с векторами. В этом блокноте разберём, как слой Embedding превращает каждый ID в обучаемый вектор чисел.

## 1. Основные понятия

- **Embedding (эмбеддинг)** — векторное представление токена.
- **Embedding dimension (`d_model`)** — количество чисел в векторе одного токена.
- **Embedding matrix** — таблица, в которой каждой строке соответствует вектор одного токена.
- **Обучаемые параметры** — числа, значения которых нейросеть меняет во время обучения.

Если в словаре 1000 токенов, а размерность эмбеддинга равна 64, таблица эмбеддингов будет иметь форму `(1000, 64)`.

## 2. Создаём маленький словарь и последовательность ID

Используем игрушечный пример, чтобы видеть каждое значение. В дальнейшем данные будут поступать из нашего датасета.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(42)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
    "the": 2,
    "movie": 3,
    "was": 4,
    "good": 5,
    "bad": 6
}

sentence_ids = torch.tensor([2, 3, 4, 5])
print("ID предложения:", sentence_ids)
print("Форма:", sentence_ids.shape)
print("Размер словаря:", len(vocab))

## 3. Таблица эмбеддингов — обычная матрица

Представим, что для каждого из 7 токенов мы храним вектор из 4 чисел. Тогда таблица имеет форму `(7, 4)`.

Пока заполним её случайными числами. В настоящей модели эти значения будут обучаться.

In [ ]:
vocab_size = len(vocab)
embedding_dim = 4

embedding_matrix = torch.randn(vocab_size, embedding_dim)

print("Матрица эмбеддингов:")
print(embedding_matrix)
print("Форма:", embedding_matrix.shape)

print("\nВектор токена 'movie':")
print(embedding_matrix[vocab["movie"]])

## 4. Получаем векторы по ID

ID токена используется как номер строки в embedding matrix. Например, если `movie` имеет ID 3, его вектор — третья строка матрицы (нумерация начинается с нуля).

В PyTorch для этого есть удобный слой `nn.Embedding`. Он хранит обучаемую таблицу и выполняет такой поиск автоматически.

In [ ]:
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)

token_vectors = embedding(sentence_ids)

print("ID токенов:", sentence_ids)
print("Векторы токенов:\n", token_vectors)
print("Форма ID:", sentence_ids.shape)
print("Форма векторов:", token_vectors.shape)

### Почему форма изменилась?

До Embedding у нас была последовательность из 4 ID: форма `(4,)`.

После Embedding каждый ID заменился вектором из 4 чисел. Получилась матрица формы `(4, 4)`:
- первая ось — 4 токена в предложении;
- вторая ось — 4 числа в каждом векторе.

Если бы мы обрабатывали batch из 8 предложений длиной 20 токенов при `embedding_dim = 64`, форма результата была бы `(8, 20, 64)`.

## 5. Почему эмбеддинги обучаются?

`nn.Embedding` — не просто словарь с заранее заданными значениями. Его веса являются параметрами нейросети. Во время обучения алгоритм меняет их так, чтобы модель лучше решала задачу.

Посмотрим, сколько обучаемых чисел находится в нашем слое.

In [ ]:
print("Число параметров:", embedding.weight.numel())
print("Форма таблицы весов:", embedding.weight.shape)
print("Требуется градиент:", embedding.weight.requires_grad)

## 6. Специальная обработка `<PAD>`

В прошлом блокноте мы добавляли `<PAD>` к коротким предложениям. Часто удобно сделать так, чтобы вектор этого токена оставался нулевым и не обучался. Для этого есть аргумент `padding_idx`.

Важно: нулевой вектор `<PAD>` сам по себе не заменяет attention mask. Когда дойдём до Attention, отдельно разберём маскирование padding-позиций.

In [ ]:
embedding_with_pad = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=vocab["<PAD>"]
)

print("Эмбеддинг <PAD>:", embedding_with_pad.weight[vocab["<PAD>"]])
print("Эмбеддинг 'movie':", embedding_with_pad.weight[vocab["movie"]])

## 7. Самостоятельные упражнения

1. Измени `embedding_dim` с 4 на 8. Как изменилась форма `token_vectors`?
2. Создай последовательность из 5 ID, например `torch.tensor([2, 3, 4, 6, 5])`, и получи её эмбеддинги.
3. Создай batch из двух предложений: `torch.tensor([[2, 3, 5], [2, 4, 6]])`. Получи эмбеддинги и запиши форму результата.
4. Посчитай вручную, сколько параметров будет у Embedding-слоя при словаре из 5000 токенов и `embedding_dim = 64`.
5. Объясни своими словами, чем token ID отличается от embedding-вектора.

Подсказка: количество параметров таблицы равно `vocab_size * embedding_dim`.

## Что мы узнали

Теперь мы умеем превращать последовательность token ID в матрицу векторов. Это представление текста уже подходит для подачи в следующие слои Transformer.

Но есть одна проблема: Embedding не сообщает модели, в каком порядке стоят токены. Для этого следующим шагом добавим позиционную информацию — **Positional Encoding**.